# Notebook 2: Direct Preference Optimization (DPO)

## What we're doing
We fine-tune the **SFT checkpoint** from Notebook 1 using DPO — teaching the model
to prefer detailed, accurate cybersecurity responses over vague or incorrect ones.

## Task
**Response quality alignment** — given a security question, the model should produce
analyst-quality responses rather than superficial ones.

## Dataset
- Primary: `walledai/CyberGuard` preference pairs
- Fallback: synthetic (chosen=detailed response, rejected=vague response)

## How DPO works
```
For each (prompt, y_w=chosen, y_l=rejected):

  log_ratio_w = log π_θ(y_w|x) - log π_ref(y_w|x)   # how much more/less than ref model?
  log_ratio_l = log π_θ(y_l|x) - log π_ref(y_l|x)

  loss = -log σ(β * (log_ratio_w - log_ratio_l))
```
This pushes `y_w` probability UP relative to the reference, and `y_l` DOWN.
β controls how far we're willing to deviate from the reference model.

**Key insight**: DPO implicitly encodes a reward model inside the policy ratio.
No RL loop, no reward model to train separately — just a supervised loss.

## CPU config
- β=0.1 (low KL penalty, allows more deviation from ref)
- max_seq_len=128, batch=1, grad_accum=8, 1 epoch
- ~150 preference pairs → expect ~45–90 min on CPU

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'trl>=0.8', 'datasets', 'accelerate', 'torch'
])
print('Dependencies ready.')

In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from datasets import Dataset
from trl import DPOTrainer, DPOConfig

from data.loader import load_dpo_dataset, train_val_split
from shared.model_utils import (
    load_model_and_tokenizer, load_saved_model,
    generate_response, LossTracker, save_model
)

os.environ['CUDA_VISIBLE_DEVICES'] = ''
torch.manual_seed(42)
print(f'PyTorch {torch.__version__} | cpu')

In [ ]:
# ── Cell 3: Load preference dataset ─────────────────────────────────────────
MAX_SAMPLES = 150

raw_data = load_dpo_dataset(max_samples=MAX_SAMPLES)
train_data, val_data = train_val_split(raw_data, val_ratio=0.1)

print(f'\nTrain pairs: {len(train_data)} | Val pairs: {len(val_data)}')
print('\n--- Sample ---')
s = train_data[0]
print('PROMPT:   ', s['prompt'][:100])
print('CHOSEN:   ', s['chosen'][:120])
print('REJECTED: ', s['rejected'][:120])

In [ ]:
# ── Cell 4: Convert to HuggingFace Dataset ───────────────────────────────────
# DPOTrainer expects columns: 'prompt', 'chosen', 'rejected'
train_hf = Dataset.from_list(train_data)
val_hf   = Dataset.from_list(val_data)

print('Columns:', train_hf.column_names)
print('Sample prompt length:', len(train_hf[0]['prompt'].split()))

In [ ]:
# ── Cell 5: Load reference (SFT) model ───────────────────────────────────────
# DPO needs BOTH a policy model (being trained) and a reference model (frozen SFT).
# TRL DPOTrainer handles this — we just pass the base model; it auto-creates ref copy.

sft_model_path = PROJECT_ROOT / 'models' / 'sft_cybersec'

if sft_model_path.exists():
    print('Loading SFT checkpoint as starting point for DPO...')
    model, tokenizer = load_saved_model('sft_cybersec')
else:
    print('SFT checkpoint not found — loading base Qwen2.5-0.5B.')
    print('(Run 01_sft.ipynb first for best results)')
    model, tokenizer = load_model_and_tokenizer()

print(f'Policy model loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params')

In [ ]:
# ── Cell 6: Inspect the DPO loss manually (educational) ──────────────────────
#
# This cell shows EXACTLY what DPO computes for one sample, so you
# can verify your understanding of the math.
#
# DPO objective:
#   L = -log σ( β * [ log(π_θ(y_w|x)/π_ref(y_w|x)) - log(π_θ(y_l|x)/π_ref(y_l|x)) ] )
#
# Since we're using the same model for both policy and ref initially,
# the ratio starts at 1 (log=0), so initial loss ≈ log(2) ≈ 0.693.

import torch.nn.functional as F

@torch.no_grad()
def compute_sequence_logprob(model, tokenizer, prompt, response, max_len=128):
    """Compute log P(response | prompt) summed over response tokens."""
    full = prompt + ' ' + response
    prompt_ids = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=max_len).input_ids
    full_ids   = tokenizer(full,   return_tensors='pt', truncation=True, max_length=max_len).input_ids
    
    if full_ids.shape[1] <= prompt_ids.shape[1]:
        return torch.tensor(0.0)
    
    logits = model(full_ids).logits[:, :-1, :]       # (1, T-1, V)
    log_probs = F.log_softmax(logits, dim=-1)        # (1, T-1, V)
    
    # Only take log probs of response tokens
    prompt_len = prompt_ids.shape[1]
    response_ids = full_ids[:, prompt_len:]           # response token ids
    response_log_probs = log_probs[:, prompt_len-1:prompt_len-1+response_ids.shape[1], :]
    
    # Gather log probs for actual tokens
    token_log_probs = response_log_probs.gather(
        2, response_ids[:, :response_log_probs.shape[1]].unsqueeze(-1)
    ).squeeze(-1)
    return token_log_probs.sum().item()

sample = train_data[0]
model.eval()

lp_chosen   = compute_sequence_logprob(model, tokenizer, sample['prompt'], sample['chosen'])
lp_rejected = compute_sequence_logprob(model, tokenizer, sample['prompt'], sample['rejected'])

beta = 0.1
# With same model as ref, log_ratio = 0 initially
# Manual DPO loss at initialization:
import math
log_sigma = lambda x: -math.log(1 + math.exp(-x))
margin = beta * (lp_chosen - lp_rejected)   # simplification: same ref
approx_loss = -log_sigma(margin)

print('=== DPO sanity check ===')
print(f'log P(chosen):   {lp_chosen:.3f}')
print(f'log P(rejected): {lp_rejected:.3f}')
print(f'margin (β=0.1):  {margin:.3f}')
print(f'approx DPO loss: {approx_loss:.3f}  (expected ≈0.693 at initialization)')
print(f'\nInterpretation:')
print(f'  chosen log-prob {">"+str(round(lp_chosen,1)) if lp_chosen > lp_rejected else "<" + str(round(lp_chosen,1))} rejected — model currently {"prefers chosen" if lp_chosen > lp_rejected else "prefers rejected (DPO will fix this)"}')

In [ ]:
# ── Cell 7: Configure & run DPO training ─────────────────────────────────────
OUTPUT_DIR = str(PROJECT_ROOT / 'models' / 'dpo_checkpoint')

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    # ── DPO-specific ──
    beta=0.1,                    # KL regularization strength
    max_length=128,
    max_prompt_length=64,
    # ── CPU training ──
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-6,          # lower LR than SFT — fine-grained preference tuning
    warmup_ratio=0.1,
    # ── Logging ──
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=20,
    save_steps=100,
    # ── Misc ──
    fp16=False,
    bf16=False,
    report_to='none',
    dataloader_num_workers=0,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,   # TRL auto-creates frozen copy of model as reference
    args=dpo_config,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    tokenizer=tokenizer,
)

print('Starting DPO training...')
print(f'  β (KL weight): {dpo_config.beta}')
print(f'  Pairs: {len(train_hf)} train, {len(val_hf)} val')
print('  Each step: forward pass through BOTH policy and reference model on chosen+rejected\n')

train_result = dpo_trainer.train()

print('\n=== DPO training complete ===')
print(f'  Final loss:    {train_result.training_loss:.4f}')
print(f'  Train time:    {train_result.metrics["train_runtime"]:.0f}s')

In [ ]:
# ── Cell 8: Analyze preference margins ───────────────────────────────────────
# After training, check if the model now assigns higher prob to chosen over rejected

print('=== Post-DPO preference analysis ===')
print('Checking if model now prefers chosen > rejected on val samples...\n')

model.eval()
wins, losses = 0, 0
margins = []

for sample in val_data[:10]:
    lp_w = compute_sequence_logprob(model, tokenizer, sample['prompt'], sample['chosen'])
    lp_l = compute_sequence_logprob(model, tokenizer, sample['prompt'], sample['rejected'])
    margin = lp_w - lp_l
    margins.append(margin)
    if margin > 0:
        wins += 1
    else:
        losses += 1
    print(f'  chosen logP={lp_w:.2f} | rejected logP={lp_l:.2f} | margin={margin:+.2f} | {"WIN" if margin > 0 else "LOSS"}')

print(f'\n  Win rate: {wins}/{wins+losses} = {wins/(wins+losses)*100:.0f}%')
print(f'  Avg margin: {sum(margins)/len(margins):.3f}')
print('  (Higher win rate = DPO successfully re-ranked preferences)')

In [ ]:
# ── Cell 9: Qualitative comparison ───────────────────────────────────────────
test_prompt = (
    'You are a cybersecurity expert. Explain the following:\n\n'
    'What is the difference between IDS and IPS?\n\nAnswer:'
)

print('=== DPO model response ===')
resp = generate_response(model, tokenizer, test_prompt, max_new_tokens=100)
print(resp)

print('\n=== What DPO trained us to prefer ===')
print('CHOSEN (target quality):')
print(train_data[3]['chosen'][:300])
print('\nREJECTED (what we move away from):')
print(train_data[3]['rejected'][:150])

In [ ]:
# ── Cell 10: Save ─────────────────────────────────────────────────────────────
save_model(model, tokenizer, 'dpo_cybersec')

tracker = LossTracker()
for log in dpo_trainer.state.log_history:
    if 'loss' in log and 'step' in log:
        tracker.record(log['step'], log['loss'])

loss_path = str(PROJECT_ROOT / 'models' / 'dpo_loss_history.json')
tracker.save(loss_path)
tracker.plot_ascii('DPO Training Loss')

print('Notebook 2 complete. Model saved to models/dpo_cybersec')

## Key takeaways

1. **DPO needs a reference model**: the loss measures how far the policy deviates from the SFT baseline. Without a good SFT starting point, DPO has nothing to anchor to.

2. **β controls the tradeoff**: small β → model can drift further from reference (more aggressive preference learning). Large β → stays conservative.

3. **No RL**: DPO is a supervised loss computed entirely in one forward pass per batch. Much more stable than PPO.

4. **The implicit reward**: the DPO-trained model implicitly encodes `r(x,y) = β * log(π_θ(y|x) / π_ref(y|x))`. You can extract this as a reward signal for downstream use.

5. **Next**: `03_grpo.ipynb` — GRPO uses *verifiable* rewards (is the CVE severity correct?) instead of preference pairs.